In [1]:
# Step 0: Environment Setup and Library Imports
# ------------------------------------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-GUI backend for training visualizations
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import warnings
import os
import time
import pickle

# Enable anomaly detection to get more detailed error messages if NaNs appear
torch.autograd.set_detect_anomaly(True)
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    num_gpus = torch.cuda.device_count()
    print(f"Number of GPUs available: {num_gpus}")
    for i in range(num_gpus):
        print(f"GPU Name {i}: {torch.cuda.get_device_name(i)}")
    torch.cuda.empty_cache()

PyTorch Version: 2.5.1
Using device: cuda
Number of GPUs available: 2
GPU Name 0: NVIDIA GeForce GTX 1080 Ti
GPU Name 1: NVIDIA GeForce GTX 1080 Ti


In [5]:
# Step 1: Data Loading & Initial Preprocessing
# ------------------------------------------------------------------------------
training_data_path1 = r'dAtA/cmems_mod_ibi_wav_my_0.027deg_PT1H-i_multi-vars_11.00W-8.53W_38.50N-40.47N_2020-01-01-2023-12-30.nc'
training_data_path_static = r'dAtA/cmems_mod_ibi_wav_my_0.027deg_static_1756186965833.nc'

print(f"\n--- Loading and casting raw data into RAM ---")

# Cast data to float32 to halve memory usage, then load into RAM.
ds_static = xr.open_dataset(training_data_path_static, engine="netcdf4").astype(np.float32).load()
ds1 = xr.open_dataset(training_data_path1, engine="netcdf4").astype(np.float32).load()
print("Data successfully loaded into memory as float32.")

static_mask = ds_static['mask'].values
static_depth = ds_static['deptho'].values

# Split data temporally
ds_train = ds1.sel(time=slice('2020-01-01', '2022-12-31'))
ds_val = ds1.sel(time=slice('2023-01-01', '2023-06-30'))
ds_test = ds1.sel(time=slice('2023-07-01', '2023-12-30'))

# Feature engineering
ds_train['VSDmag'] = np.sqrt(ds_train['VSDX']**2 + ds_train['VSDY']**2)
ds_val['VSDmag'] = np.sqrt(ds_val['VSDX']**2 + ds_val['VSDY']**2)
ds_test['VSDmag'] = np.sqrt(ds_test['VSDX']**2 + ds_test['VSDY']**2)

# Define variables
target_var = 'VCMX'
initial_feature_vars = ['VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
all_vars_to_process = [target_var] + initial_feature_vars

print(f"Target variable: {target_var}")
print(f"Feature variables: {initial_feature_vars}")


--- Loading and casting raw data into RAM ---
Data successfully loaded into memory as float32.
Target variable: VCMX
Feature variables: ['VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']


In [6]:
# Step 2: Fit Scalers on Training Data Only
# ------------------------------------------------------------------------------
print("\n--- Fitting scalers on TRAINING data only... ---")
scalers = {}
chunk_size = 1000

for var in all_vars_to_process:
    scaler = MinMaxScaler()
    # Since data is in memory, we can fit all at once without chunking
    data_to_fit = ds_train[var].values.reshape(-1, 1)
    scaler.fit(data_to_fit)
    scalers[var] = scaler
    print(f"Scaler fitted for {var}")

# Process static data
static_scalers = {}
static_scalers['deptho'] = MinMaxScaler()
static_depth_scaled = static_scalers['deptho'].fit_transform(static_depth.reshape(-1, 1)).reshape(static_depth.shape)

# Correctly stack both the mask and the scaled depth
static_data_tensor = np.stack([static_mask, static_depth_scaled], axis=0).astype(np.float32)

print("\nScalers fitted successfully.")
print(f"Static data tensor shape: {static_data_tensor.shape}")


--- Fitting scalers on TRAINING data only... ---
Scaler fitted for VCMX
Scaler fitted for VSDmag
Scaler fitted for VTM10
Scaler fitted for VTM02
Scaler fitted for VTM01_WW
Scaler fitted for VTM01_SW1
Scaler fitted for VMXL
Scaler fitted for VHM0_WW
Scaler fitted for VHM0_SW1

Scalers fitted successfully.
Static data tensor shape: (2, 72, 90)


In [8]:
# Step 3: Save All Processed Data
# ------------------------------------------------------------------------------
import pickle
import numpy as np
import os

print("\n--- Saving all processed data to disk for future use... ---")

# FIX: Changed the path to your D: drive
processed_data_dir = r'D:\babe_prediction'
os.makedirs(processed_data_dir, exist_ok=True)
print(f"Data will be saved in: {processed_data_dir}")

# 1. Save the scalers dictionary using pickle
with open(os.path.join(processed_data_dir, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)

# 2. Save the static data tensor numpy array
np.save(os.path.join(processed_data_dir, 'static_data_tensor.npy'), static_data_tensor)

# 3. Save the processed and split xarray datasets
ds_train.to_netcdf(os.path.join(processed_data_dir, 'ds_train.nc'))
ds_val.to_netcdf(os.path.join(processed_data_dir, 'ds_val.nc'))
ds_test.to_netcdf(os.path.join(processed_data_dir, 'ds_test.nc'))

print(f"✅ All processed data has been saved to the '{processed_data_dir}' directory.")


--- Saving all processed data to disk for future use... ---
Data will be saved in: D:\babe_prediction
✅ All processed data has been saved to the 'D:\babe_prediction' directory.


------------------------START THE CODE RUNNING FROM HERE IF THE SCALERS ALREADY LOADED, IGONRE ABOVE CODE -------

## ➡️ Future Sessions: Start Here!
For all future runs, you can skip Cells 2 and 3 and instead run the cell below to load everything in seconds

In [5]:
# --- Quick Load for Future Sessions ---
import pickle
import numpy as np
import xarray as xr
import os
import torch

print("--- Loading pre-processed data quickly from disk... ---")

# FIX: Changed the path to your D: drive to match where you saved it
processed_data_dir = r'D:\babe_prediction'

# 1. Load scalers, static tensor, and datasets
print(f"Loading data from: {processed_data_dir}")
with open(os.path.join(processed_data_dir, 'scalers.pkl'), 'rb') as f:
    scalers = pickle.load(f)
static_data_tensor = np.load(os.path.join(processed_data_dir, 'static_data_tensor.npy'))
ds_train = xr.open_dataset(os.path.join(processed_data_dir, 'ds_train.nc')).load()
ds_val = xr.open_dataset(os.path.join(processed_data_dir, 'ds_val.nc')).load()
ds_test = xr.open_dataset(os.path.join(processed_data_dir, 'ds_test.nc')).load()

# 2. Re-define necessary variables
target_var = 'VCMX'
initial_feature_vars = ['VSDmag', 'VTM10', 'VTM02', 'VTM01_WW', 'VTM01_SW1', 'VMXL', 'VHM0_WW', 'VHM0_SW1']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ All data loaded in seconds. Using device: {device}")

--- Loading pre-processed data quickly from disk... ---
Loading data from: D:\babe_prediction
✅ All data loaded in seconds. Using device: cuda


In [ ]:
# Step 4: PyTorch Dataset and DataLoaders
# ------------------------------------------------------------------------------
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time
import os

class WaveDataset(Dataset):
    def __init__(self, ds, feature_vars, target_var, scalers, static_data_tensor, lookback=48, forecast_horizon=168):
        self.ds = ds
        self.feature_vars = feature_vars
        self.target_var = target_var
        self.scalers = scalers
        self.static_data_tensor = static_data_tensor
        self.lookback = lookback
        self.forecast_horizon = forecast_horizon
        self.n_sequences = len(self.ds['time']) - (self.lookback + self.forecast_horizon)
        # Get the process ID for the print statement
        self.worker_id = None

    def __len__(self):
        return self.n_sequences

    def __getitem__(self, idx):
        # FIX: Added a print statement to show that the worker process is active.
        # This will print once for each sample being prepared in the first batch.
        # -------------------------------------------------------------------
        if self.worker_id is None:
            self.worker_id = os.getpid()
        if idx < 8: # Only print for the first batch to avoid clutter
            print(f"[Worker PID: {self.worker_id}] Preparing sample #{idx}...")
        # -------------------------------------------------------------------

        seq_end_idx = idx + self.lookback + self.forecast_horizon
        data_slice = self.ds.isel(time=slice(idx, seq_end_idx))
        
        scaled_data = {}
        for var, scaler in self.scalers.items():
            if var not in data_slice.variables: continue
            scaled_data[var] = scaler.transform(data_slice[var].values.reshape(-1, 1)).reshape(data_slice[var].shape)

        features_list = [
            scaled_data[var][:self.lookback, np.newaxis, :, :] 
            for var in self.feature_vars
        ]
        
        static_data_with_time = np.tile(self.static_data_tensor[np.newaxis, ...], (self.lookback, 1, 1, 1))
        all_features = np.concatenate(features_list + [static_data_with_time], axis=1).astype(np.float32)
        y = scaled_data[self.target_var][self.lookback:].astype(np.float32)

        return torch.from_numpy(all_features), torch.from_numpy(y)

# --- Create DataLoaders ---
lookback_hours = 48
forecast_hours = 168
batch_size = 8

train_dataset = WaveDataset(ds_train, initial_feature_vars, target_var, scalers, static_data_tensor)
val_dataset = WaveDataset(ds_val, initial_feature_vars, target_var, scalers, static_data_tensor)
test_dataset = WaveDataset(ds_test, initial_feature_vars, target_var, scalers, static_data_tensor)

num_workers_to_use = 1 
print(f"Using {num_workers_to_use} dedicated CPU core for stable data loading.")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers_to_use, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers_to_use, pin_memory=True, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers_to_use, pin_memory=True, persistent_workers=True)

print("DataLoaders instantiated. Checking first batch (this may take a minute)...")

# FIX: Added a timer to measure the one-time startup cost.
# -------------------------------------------------------------------
start_time = time.time()
sample_X, sample_y = next(iter(train_loader))
end_time = time.time()
# -------------------------------------------------------------------

print(f"\n✅ First batch loaded successfully in {end_time - start_time:.2f} seconds.")
print(f"Sample X shape: {sample_X.shape}")
print(f"Sample y shape: {sample_y.shape}")

Using 1 dedicated CPU core for stable data loading.
DataLoaders instantiated. Checking first batch (this may take a minute)...


In [ ]:
# Step 5: ConvLSTM Model Definition
# ------------------------------------------------------------------------------
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, bias):
        super(ConvLSTMCell, self).__init__()
        self.input_dim, self.hidden_dim = input_dim, hidden_dim
        self.kernel_size, self.bias = kernel_size, bias
        self.padding = kernel_size[0] // 2
        self.conv = nn.Conv2d(self.input_dim + self.hidden_dim, 4 * self.hidden_dim, self.kernel_size, padding=self.padding, bias=self.bias)
    
    def forward(self, x, h_c):
        h, c = h_c
        combined = torch.cat([x, h], dim=1)
        cc = self.conv(combined)
        i, f, o, g = torch.split(cc, self.hidden_dim, dim=1)
        i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
        c_n = f * c + i * g
        h_n = o * torch.tanh(c_n)
        return h_n, c_n

class ConvLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size, num_layers, batch_first=True, bias=True):
        super(ConvLSTM, self).__init__()
        self.batch_first, self.num_layers = batch_first, num_layers
        hidden_dims = [hidden_dim] * num_layers if isinstance(hidden_dim, int) else hidden_dim
        
        cell_list = []
        for i in range(self.num_layers):
            cur_input_dim = input_dim if i == 0 else hidden_dims[i - 1]
            cell_list.append(ConvLSTMCell(cur_input_dim, hidden_dims[i], kernel_size, bias))
        self.cell_list = nn.ModuleList(cell_list)

    def forward(self, x, h_c=None):
        if not self.batch_first: x = x.permute(1, 0, 2, 3, 4)
        b, s_l, _, h, w = x.size()
        if h_c is None: h_c = self._init_hidden(b, h, w, x.device)
        
        cur_in = x
        for l_idx in range(self.num_layers):
            h, c = h_c[l_idx]
            output_inner = []
            for t in range(s_l):
                h, c = self.cell_list[l_idx](cur_in[:, t, :, :, :], [h, c])
                output_inner.append(h)
            cur_in = torch.stack(output_inner, dim=1)
        return cur_in, [h, c]

    def _init_hidden(self, b, h, w, d):
        return [cell.init_hidden(b, h, w, d) for cell in self.cell_list]

class ConvLSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], kernel_size=(3, 3)):
        super(ConvLSTMNet, self).__init__()
        self.cl1 = ConvLSTM(input_dim, hidden_dims[0], kernel_size, 1, batch_first=True)
        self.cl2 = ConvLSTM(hidden_dims[0], hidden_dims[1], kernel_size, 1, batch_first=True)
        self.output_conv = nn.Conv2d(hidden_dims[1], 1, kernel_size=(1, 1), padding='same')

    def forward(self, x_seq):
        l1_o, _ = self.cl1(x_seq)
        l2_o, _ = self.cl2(l1_o)
        return self.output_conv(l2_o[:, -1, :, :, :])

# --- Create Model ---
n_feature_channels = len(initial_feature_vars)
n_static_channels = static_data_tensor.shape[0]
input_channels = n_feature_channels + n_static_channels

model = ConvLSTMNet(input_dim=input_channels)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for training!")
    model = nn.DataParallel(model)
model.to(device)
print(f"Model built. Total Input Channels: {input_channels}")

In [ ]:
# Step 6: Training Functions
# ------------------------------------------------------------------------------
def weighted_mse_loss(output, target, weight_threshold=0.8):
    mse = (output - target)**2
    weights = torch.where(target > weight_threshold, 2.0, 1.0)
    return torch.mean(weights * mse)

def train_model(model, train_loader, val_loader, device, epochs=50, model_path='best_convlstm_model.pth'):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
    loss_fn = weighted_mse_loss
    best_val_loss = float('inf')
    scaler = torch.cuda.amp.GradScaler()
    patience = 5
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    print("\n--- Starting Model Training ---")
    
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        total_train_loss = 0.0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Training]")
        
        for X, y in train_pbar:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                predicted_step = model(X)
                target_step = y[:, 0, :, :].unsqueeze(1)
                loss = loss_fn(predicted_step, target_step)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            total_train_loss += loss.item()
            train_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        
        avg_train_loss = total_train_loss / len(train_loader)
        
        model.eval()
        total_val_loss = 0.0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Validation]")
        
        with torch.no_grad():
            for X, y in val_pbar:
                X, y = X.to(device), y.to(device)
                with torch.cuda.amp.autocast():
                    predicted_step = model(X)
                    target_step = y[:, 0, :, :].unsqueeze(1)
                    loss = loss_fn(predicted_step, target_step)
                total_val_loss += loss.item()
                val_pbar.set_postfix({'loss': f'{loss.item():.6f}'})
        
        avg_val_loss = total_val_loss / len(val_loader)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        
        epoch_time = time.time() - start_time
        print(f"Epoch {epoch+1}/{epochs} - {epoch_time:.1f}s - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            model_state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(model_state, model_path)
            print(f"✅ New best model saved with validation loss: {best_val_loss:.6f}")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
    
    best_model_state = torch.load(model_path)
    if isinstance(model, nn.DataParallel): model.module.load_state_dict(best_model_state)
    else: model.load_state_dict(best_model_state)
    print(f"\n✅ Training completed. Best model loaded from {model_path}")
    return model, history

print("Training functions defined successfully.")

In [ ]:
# Step 7: Start Training
# ------------------------------------------------------------------------------
trained_model, training_history = train_model(model, train_loader, val_loader, device, epochs=50)

# --- Plot training history ---
plt.figure(figsize=(12, 6))
plt.plot(training_history['train_loss'], label='Training Loss')
plt.plot(training_history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Training run finished and history plot saved.")

In [ ]:
# Step 8: Model Evaluation
# ------------------------------------------------------------------------------
def evaluate_model(model, test_loader, device, scalers, target_var):
    model.eval()
    all_predictions, all_targets = [], []
    
    print("\n--- Evaluating Model on Test Data ---")
    
    with torch.no_grad():
        for X, y in tqdm(test_loader, desc="Evaluating"):
            X, y = X.to(device), y.to(device)
            predictions = model(X)
            
            predictions_np = predictions.cpu().numpy()
            targets_np = y[:, 0, :, :].cpu().numpy()
            
            all_predictions.append(predictions_np)
            all_targets.append(targets_np)
    
    predictions = np.concatenate(all_predictions, axis=0)
    targets = np.concatenate(all_targets, axis=0)
    
    target_scaler = scalers[target_var]
    pred_original = target_scaler.inverse_transform(predictions.reshape(-1, 1)).reshape(predictions.shape)
    target_original = target_scaler.inverse_transform(targets.reshape(-1, 1)).reshape(targets.shape)
    
    mse = np.mean((pred_original - target_original)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(pred_original - target_original))
    
    print(f"\n📊 Test Results:")
    print(f"  MSE: {mse:.6f}")
    print(f"  RMSE: {rmse:.6f}")
    print(f"  MAE: {mae:.6f}")
    
    return pred_original, target_original, {'mse': mse, 'rmse': rmse, 'mae': mae}

# --- Evaluate the final model ---
predictions, targets, metrics = evaluate_model(trained_model, test_loader, device, scalers, target_var)

In [ ]:
# Step 10: Explainable AI (XAI) with Grad-CAM
# ------------------------------------------------------------------------------
print("\n--- Generating Grad-CAM Explanation ---")

# Use the trained model and put it in evaluation mode
model_to_explain = trained_model
model_to_explain.eval()

# Handle DataParallel wrapper if it exists
if isinstance(model_to_explain, nn.DataParallel):
    model_to_explain = model_to_explain.module

# Define the target layer for Grad-CAM (last convolutional layer is a good choice)
target_layer = model_to_explain.cl2.cell_list[0].conv

# Get a sample batch from the test data to explain
X_sample, _ = next(iter(test_loader))
X_sample = X_sample.to(device)

# Grad-CAM expects a 4D tensor (B, C, H, W). We'll explain the last time step of the input sequence.
# Let's take the first item of the batch to visualize.
input_tensor = X_sample[0:1, -1, :, :, :] # Shape: (1, C, H, W)

# The feature map we'll overlay the heatmap on (e.g., wind speed, channel 1)
# We need to un-normalize it to see the real values.
vtm10_scaler = scalers['VTM10']
input_feature_map = X_sample[0, -1, 1, :, :].cpu().numpy()
input_feature_map_original = vtm10_scaler.inverse_transform(input_feature_map.reshape(-1, 1)).reshape(input_feature_map.shape)

# Instantiate GradCAM
cam = GradCAM(model=model_to_explain, target_layer=target_layer)

# Generate the heatmap. For regression, we don't need a target class.
grayscale_cam = cam(input_tensor=input_tensor, target_category=None)
grayscale_cam = grayscale_cam[0, :] # Take the first (and only) image

# Visualize the heatmap and overlay it on the input feature map
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
axes[0].imshow(input_feature_map_original, cmap='coolwarm')
axes[0].set_title('Input Feature (Wind Speed)')
axes[1].imshow(grayscale_cam, cmap='jet')
axes[1].set_title('Grad-CAM Heatmap')

# Overlay
visualization = show_cam_on_image(input_feature_map_original / np.max(input_feature_map_original), grayscale_cam, use_rgb=False)
axes[2].imshow(visualization)
axes[2].set_title('Model Focus Overlay')

plt.tight_layout()
plt.savefig('grad_cam_explanation.png', dpi=300)
plt.show()

print("✅ Grad-CAM visualization saved.")

In [ ]:
# Step 11: Explainable AI (XAI) with SHAP
# ------------------------------------------------------------------------------
print("\n--- Generating SHAP Explanation (This may take a while)... ---")

# 1. Get a background dataset (for the explainer's baseline) from the training set
background_batch, _ = next(iter(train_loader))
background_batch = background_batch.to(device)

# 2. Get a few samples to explain from the test set
test_samples, _ = next(iter(test_loader))
test_samples = test_samples[0:2].to(device) # Explain only 2 samples to save time

# 3. Create the SHAP GradientExplainer
# Use the unwrapped model if using DataParallel
model_to_explain = trained_model.module if isinstance(trained_model, nn.DataParallel) else trained_model
explainer = shap.GradientExplainer(model_to_explain, background_batch)

# 4. Calculate SHAP values for the test samples
shap_values = explainer.shap_values(test_samples)

print("SHAP values calculated. Now creating plot.")

# 5. Visualize the results
# To get global feature importance, we average the absolute SHAP values across all dimensions except the channel dimension
mean_abs_shap = np.mean(np.abs(shap_values), axis=(0, 1, 3, 4))

# Create the full list of feature names in the correct order
full_feature_names = initial_feature_vars + ['mask', 'deptho']

# Plot the feature importance
plt.figure(figsize=(10, 8))
y_pos = np.arange(len(full_feature_names))
plt.barh(y_pos, mean_abs_shap, align='center')
plt.yticks(y_pos, labels=full_feature_names)
plt.gca().invert_yaxis()  # Most important on top
plt.xlabel('Mean Absolute SHAP Value (Impact on Model Output)')
plt.title('SHAP Global Feature Importance')
plt.grid(axis='x')
plt.savefig('shap_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ SHAP feature importance plot saved.")

In [ ]:
# Step 9: Visualization
# ------------------------------------------------------------------------------
def plot_predictions(predictions, targets, sample_idx=0, save_path='prediction_comparison.png'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    pred_sample = predictions[sample_idx].squeeze()
    targ_sample = targets[sample_idx].squeeze()
    
    v_max = max(np.max(pred_sample), np.max(targ_sample))
    
    im1 = axes[0].imshow(targ_sample, cmap='viridis', vmin=0, vmax=v_max)
    axes[0].set_title(f'Target (Ground Truth) - Sample {sample_idx}')
    plt.colorbar(im1, ax=axes[0], orientation='horizontal', pad=0.2)
    
    im2 = axes[1].imshow(pred_sample, cmap='viridis', vmin=0, vmax=v_max)
    axes[1].set_title(f'Model Prediction - Sample {sample_idx}')
    plt.colorbar(im2, ax=axes[1], orientation='horizontal', pad=0.2)
    
    diff = pred_sample - targ_sample
    diff_max = np.max(np.abs(diff))
    im3 = axes[2].imshow(diff, cmap='RdBu_r', vmin=-diff_max, vmax=diff_max)
    axes[2].set_title(f'Difference (Pred - Target) - Sample {sample_idx}')
    plt.colorbar(im3, ax=axes[2], orientation='horizontal', pad=0.2)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# --- Plot some sample predictions ---
print("\n--- Generating Visualization Plots ---")
plot_predictions(predictions, targets, sample_idx=15)
plot_predictions(predictions, targets, sample_idx=50, save_path='prediction_comparison_2.png')

print("\n🎉 Ocean Wave Height Prediction Pipeline Completed Successfully!")